In [3]:
# Final brandequity
import os
import pandas as pd
from pyspark.sql import SparkSession

input_folder = "/lakehouse/default/Files/Beast"

sheets = [
    "Total Sample","SS7_Caucasian","SS2_Main_ African American","SS2_African American",
    "SS1_Main_ Hispanic","SS1_Hispanic","SS3_Gen Z (27 and under)","SS3_Gen Z (28 and under)",
    "SS3_Gen Z (29 and under)","SS4_Millennials (28-43)","SS4_Millennials (29-44)",
    "SS4_Millennials (30-45)","SS5_Gen X (44-59)","SS5_Gen X (45-60)","SS5_Gen X (46-61)",
    "SS6_Boomers (60+)","SS6_Boomers (61+)","SS6_Boomers (62+)","SS7_Males 11-17",
    "SS8_Males 11-17","SS8_Males 18-24","SS9_Males 18-24","SS9_Males 18-24)"
]   

main_metrics = [
    "Meaningful (indexed against first wave category average)",
    "Different (indexed against first wave category average)",
    "Salient (indexed against first wave category average)",
    "Demand Power % (share)",
    "Pricing Power (indexed vs. first wave category average)"
]

VALID_KPIS = {"Meaningful","Different","Salient","Demand Power","Pricing Power"}

master_data = []
all_files = os.listdir(input_folder)
excel_files = [f for f in all_files if f.endswith(".xlsx")]

print("Excel Files Found:", excel_files)

for file_name in excel_files:
    file_path = os.path.join(input_folder, file_name)
    print(f"\nProcessing File: {file_name}")

    for sheet_tabs in sheets:
        try:
            df = pd.read_excel(file_path, sheet_name=sheet_tabs, engine="openpyxl")
            print(f"Processed Sheet: {sheet_tabs}")
        except Exception as e:
            print(f"Skipping Sheet: {sheet_tabs} | Error: {e}")
            continue

        df = df.dropna(how='all').dropna(axis=1, how='all')
        df.rename(columns={df.columns[0]: "Metric"}, inplace=True)

        # MAIN METRICS
        df_main = df[df["Metric"].isin(main_metrics)].copy()
        df_main["Metric"] = df_main["Metric"].replace({
            "Meaningful (indexed against first wave category average)":"Meaningful Index",
            "Different (indexed against first wave category average)":"Different Index",
            "Salient (indexed against first wave category average)":"Salient Index",
            "Demand Power % (share)":"Brand Power Share",
            "Pricing Power (indexed vs. first wave category average)":"Premium Index"
        })
        main_out = df_main.melt(id_vars=["Metric"], var_name="Brand", value_name="Value")

        # SECTION 7 PARSER
        df_std = df.copy()
        start_idx = df_std[df_std["Metric"].astype(str).str.contains("Section 7", case=False, na=False)].index
        if len(start_idx) == 0:
            continue
        df_std = df_std.loc[start_idx[0] + 1:].copy()

        current_kpi = None
        rows = []

        for _, row in df_std.iterrows():
            m = str(row["Metric"]).strip().lower()

            if "section" in m and "7" not in m:
                break
            if m in ["p-value", "significant vs. avg"]:
                continue

            # Std Error row
            if m == "std error" and current_kpi:
                temp = row.copy()
                temp["Metric"] = current_kpi
                rows.append(temp)
                continue

            # KPI detection with hybrid logic
            base = m.split("(")[0].strip().lower()
            current_kpi = None
            for kpi in VALID_KPIS:
                if base.startswith(kpi.lower()):
                    if kpi in ["demand power","pricing power"]:
                        # strict: only Final or plain
                        if "raw" in m:
                            current_kpi = None
                        elif "final" in m or m == kpi.lower():
                            current_kpi = kpi
                    else:
                        # broader: allow factor scores, Final, or plain
                        if "raw" in m:
                            current_kpi = None
                        elif "factor scores" in m or "final" in m or m == kpi.lower():
                            current_kpi = kpi
                    break

        df_std_clean = pd.DataFrame(rows)
        if len(df_std_clean) == 0:
            continue
        std_out = df_std_clean.melt(id_vars=["Metric"], var_name="Brand", value_name="Std_Error")
        std_out["Metric"] = std_out["Metric"].replace({
            "Meaningful":"Meaningful Index",
            "Different":"Different Index",
            "Salient":"Salient Index",
            "Demand Power":"Brand Power Share",
            "Pricing Power":"Premium Index"
        })

        main_out = main_out.dropna(subset=["Value","Brand"])
        std_out = std_out.dropna(subset=["Std_Error","Brand"])

        final_df = pd.merge(main_out, std_out, on=["Metric","Brand"], how="outer")
        final_df["Sheet_Name"] = sheet_tabs
        final_df["File_Name"] = file_name
        master_data.append(final_df)

# FINAL CONCAT
if len(master_data) > 0:
    final_master_df = pd.concat(master_data, ignore_index=True)
    final_master_df["Country"] = final_master_df["File_Name"].str.split('_').str[0].str.upper()
    final_master_df["Category"] = final_master_df["File_Name"].str.split('_').str[1].str.upper()
    final_master_df["Country_Category_key"] = final_master_df["Country"] + "_" + final_master_df["Category"]
    final_master_df["Quarter_temp"] = final_master_df["File_Name"].str.split('_').str[2].str.upper().str.split('.').str[0]
    final_master_df["Quarter"] = final_master_df["Quarter_temp"].str[6:]+" ("+final_master_df["Quarter_temp"].str[0:2]+" '"+final_master_df["Quarter_temp"].str[4:6]+")"
    final_master_df.drop(columns=["Quarter_temp"], inplace=True)
    final_master_df = final_master_df[final_master_df["Brand"]!="Average"]
    print("Final Shape:", final_master_df.shape)
    display(final_master_df.head())
else:
    print("No data processed.")


StatementMeta(, 6af59bb4-c5c4-47f6-adee-f2eefe8b350c, 6, Finished, Available, Finished, False)

Excel Files Found: ['us_fbsn_q12025mat.xlsx', 'us_fbsn_q12026mat_aa_hispanic_ss.xlsx', 'us_fbsn_q22025mat.xlsx', 'us_fbsn_q22026mat_aa_hispanic_ss.xlsx', 'us_fbsn_q32024.xlsx', 'us_fbsn_q32025mat_sprun.xlsx', 'us_fbsn_q42024mat.xlsx', 'us_fbsn_q42025mat_aa_hispanic_ss.xlsx', 'us_fmon_q22026mat_aa_hispanic_ss.xlsx', 'us_fmsn_q12023mat_aa_hispanic_ss_local.xlsx', 'us_fmsn_q12024mat_aa_hispanic_ss_local.xlsx', 'us_fmsn_q12025mat.xlsx', 'us_fmsn_q12026mat_aa_hispanic_ss.xlsx', 'us_fmsn_q22023mat_aa_hispanic_ss_local.xlsx', 'us_fmsn_q22024mat_aa_hispanic_ss_local.xlsx', 'us_fmsn_q22025mat_aa_hispanic_ss_local.xlsx', 'us_fmsn_q22026mat_aa_hispanic_ss.xlsx', 'us_fmsn_q32023mat_aa_hispanic_ss_local.xlsx', 'us_fmsn_q32024mat_aa_hispanic_ss.xlsx', 'us_fmsn_q32025mat_aa_hispanic_ss.xlsx', 'us_fmsn_q42023mat_aa_hispanic_ss_local.xlsx', 'us_fmsn_q42024mat_aa_hispanic_ss_local.xlsx', 'us_fmsn_q42025mat_aa_hispanic_ss.xlsx', 'us_fsin_q12025mat.xlsx', 'us_fsin_q12026mat_aa_hispanic_ss.xlsx', 'us_fsin_

/home/trusted-service-user/cluster-env/trident_env/lib/python3.11/site-packages/openpyxl/worksheet/_read_only.py:79: UserWarning: Unknown extension is not supported and will be removed
  for idx, row in parser.parse():
/home/trusted-service-user/cluster-env/trident_env/lib/python3.11/site-packages/openpyxl/worksheet/_read_only.py:79: UserWarning: Conditional Formatting extension is not supported and will be removed
  for idx, row in parser.parse():


SynapseWidget(Synapse.DataFrame, fcabc5ea-9e37-483d-b2f5-ddec70cd7cda)

In [9]:
# creating table BRANDEQUITY
spark_df = spark.createDataFrame(final_master_df)

spark_df.write.mode("overwrite").saveAsTable(
    "brandequitytable"
)

print("Table Saved Successfully")

StatementMeta(, 6af59bb4-c5c4-47f6-adee-f2eefe8b350c, 20, Finished, Available, Finished, False)

Table Saved Successfully


In [6]:
# CLEANING DATA FOR BIPS
import os
import pandas as pd
from pyspark.sql import SparkSession

input_folder = "/lakehouse/default/Files/Beast"
sheets = [
"Total Sample",
"SS7_Caucasian",
"SS2_Main_ African American",
"SS2_African American",
"SS1_Main_ Hispanic",
"SS1_Hispanic",
"SS3_Gen Z (27 and under)",
"SS3_Gen Z (28 and under)",
"SS3_Gen Z (29 and under)",
"SS4_Millennials (28-43)",
"SS4_Millennials (29-44)",
"SS4_Millennials (30-45)",
"SS5_Gen X (44-59)",
"SS5_Gen X (45-60)",
"SS5_Gen X (46-61)",
"SS6_Boomers (60+)",
"SS6_Boomers (61+)",
"SS6_Boomers (62+)",
"SS7_Males 11-17",
"SS8_Males 11-17",
"SS8_Males 18-24",
"SS9_Males 18-24",
"SS9_Males 18-24)"
]
master_data = []
all_files= os.listdir(input_folder)
excel_file = [ f for f in all_files if f.endswith(".xlsx")]
# print("excel file found",excel_file)

for file_name in excel_file:
    file_path=os.path.join(input_folder,file_name)
    
    for sheettab in sheets:
        
        try:
            df=pd.read_excel(file_path,sheet_name=sheettab,engine="openpyxl")
            print(f"Processing sheet ::{sheettab}")
            
        except Exception as e:
            print(f"skipping the file:{sheettab}| error:{e}")
            continue
        df=df.dropna(how="all")
        df=df.dropna(axis=1,how="all")
        df.rename(columns={df.columns[0]:"Metric"},inplace=True)
        metric = df["Metric"].tolist()

        # Find all occurrences of Brand Image Profiles
        profile_positions = [
            i for i, x in enumerate(metric)
            if x == "Brand Image Profiles (excluding any negative statements)"
        ]

        # Final ending
        final_end = metric.index("Final factor analysis scores")
        start1 = profile_positions[0]

        # second profile heading
        end1 = profile_positions[1]
        
        # skip:
        # heading row
        # Based on row
        absolute_df1 = df.iloc[start1 + 2:end1].copy()

        absolute_df1 = absolute_df1.dropna(how="all")

        main_out1 = absolute_df1.melt(
            id_vars="Metric",
            var_name="Brand",
            value_name="value"
        )

        main_out1["Base"] = metric[start1 + 1]

        main_out1["File_Name"] = file_name
        main_out1["Sheet_Name"] = sheettab
        
        start2 = profile_positions[1]

        end2 = final_end

        absolute_df2 = df.iloc[start2 + 2:end2].copy()

        absolute_df2 = absolute_df2.dropna(how="all")

        main_out2 = absolute_df2.melt(
            id_vars="Metric",
            var_name="Brand",
            value_name="value"
        )

        main_out2["Base"] = metric[start2 + 1]

        main_out2["File_Name"] = file_name
        main_out2["Sheet_Name"] = sheettab
        
        main_out=pd.concat([main_out1,main_out2],ignore_index=True)
        
        master_data.append(main_out)
if len(master_data)>0:
    final_master_data=pd.concat(master_data,ignore_index=True)
    final_master_data=final_master_data.dropna()
    final_master_data=final_master_data[final_master_data["Brand"]!="Average"]
    final_master_data["Country"]=final_master_data["File_Name"].str.split('_').str[0].str.upper()
    final_master_data["Category"]=final_master_data["File_Name"].str.split('_').str[1].str.upper()
    final_master_data["Country_Category_key"]=final_master_data["Country"]+"_"+final_master_data["Category"]
    final_master_data["temp"]=final_master_data["File_Name"].str.split('_').str[2].str.upper()
    final_master_data["Quarter"]=final_master_data["temp"].str[6:9]+" ("+ final_master_data["temp"].str[0:2] +" '"+final_master_data["temp"].str[4:6] + ")"
    final_master_data.drop(columns=["temp"], inplace=True)
    print("Shape:",final_master_data.shape)
    # output_path = os.path.join(input_folder, "BIPs.xlsx")
    print(final_master_data.head())
    # final_master_data.to_excel(
    # output_path,
    # index=False)

    print("File Saved")
   
else:
    print("No data Processed")
    


StatementMeta(, 6af59bb4-c5c4-47f6-adee-f2eefe8b350c, 16, Finished, Available, Finished, False)

Processing sheet ::Total Sample
skipping the file:SS7_Caucasian| error:Worksheet named 'SS7_Caucasian' not found
skipping the file:SS2_Main_ African American| error:Worksheet named 'SS2_Main_ African American' not found
skipping the file:SS2_African American| error:Worksheet named 'SS2_African American' not found
skipping the file:SS1_Main_ Hispanic| error:Worksheet named 'SS1_Main_ Hispanic' not found
skipping the file:SS1_Hispanic| error:Worksheet named 'SS1_Hispanic' not found
skipping the file:SS3_Gen Z (27 and under)| error:Worksheet named 'SS3_Gen Z (27 and under)' not found
skipping the file:SS3_Gen Z (28 and under)| error:Worksheet named 'SS3_Gen Z (28 and under)' not found
skipping the file:SS3_Gen Z (29 and under)| error:Worksheet named 'SS3_Gen Z (29 and under)' not found
skipping the file:SS4_Millennials (28-43)| error:Worksheet named 'SS4_Millennials (28-43)' not found
skipping the file:SS4_Millennials (29-44)| error:Worksheet named 'SS4_Millennials (29-44)' not found
skip

/home/trusted-service-user/cluster-env/trident_env/lib/python3.11/site-packages/openpyxl/worksheet/_read_only.py:79: UserWarning: Unknown extension is not supported and will be removed
  for idx, row in parser.parse():
/home/trusted-service-user/cluster-env/trident_env/lib/python3.11/site-packages/openpyxl/worksheet/_read_only.py:79: UserWarning: Conditional Formatting extension is not supported and will be removed
  for idx, row in parser.parse():
/home/trusted-service-user/cluster-env/trident_env/lib/python3.11/site-packages/openpyxl/worksheet/_read_only.py:79: UserWarning: Unknown extension is not supported and will be removed
  for idx, row in parser.parse():
/home/trusted-service-user/cluster-env/trident_env/lib/python3.11/site-packages/openpyxl/worksheet/_read_only.py:79: UserWarning: Conditional Formatting extension is not supported and will be removed
  for idx, row in parser.parse():


In [8]:
# creating table
spark_bips = spark.createDataFrame(final_master_data)

spark_bips.write.mode("overwrite").saveAsTable(
    "BIPSTable"
)

print(" BIPSTable Saved Successfully")

StatementMeta(, 6af59bb4-c5c4-47f6-adee-f2eefe8b350c, 19, Finished, Available, Finished, False)

 BIPSTable Saved Successfully
